# AmsiNet - Step 21: one consistent CPU latency column (about 8 minutes)
Run on a **CPU runtime**, *Run all*, and paste the table back. Every model is timed in this one environment (1 thread, batch 1, fp32, 10-class head, 10 warm-up and 50 timed runs, median). Line agent is timed at its real input size 64x512; all others at 224x224.

In [ ]:
import time, os, numpy as np, pandas as pd, torch
try: import timm
except ImportError:
    !pip -q install timm
    import timm
print('CPU model:', [l.split(':',1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')][0])
print('logical CPUs:', os.cpu_count(), '| torch', torch.__version__, '| threads', torch.get_num_threads())
torch.set_grad_enabled(False)
C = 10
M = [('Word agent E (EfficientNet-B0, linear)', 'efficientnet_b0.ra_in1k', {}, (224,224)),
     ('Word agent T (EfficientViT-B0, linear)', 'efficientvit_b0.r224_in1k', {}, (224,224)),
     ('Line agent (ViT-S, 64x512)', 'vit_small_patch16_224.augreg_in21k', {'img_size': (64,512)}, (64,512)),
     ('Block agent (ConvNeXt-T)', 'convnext_tiny.fb_in22k', {}, (224,224)),
     ('Page agent (MobileViT-XS)', 'mobilevit_xs.cvnets_in1k', {}, (224,224)),
     ('Flat ViT-S/16', 'vit_small_patch16_224.augreg_in21k', {}, (224,224)),
     ('Flat ViT-B/16', 'vit_base_patch16_224.augreg_in21k', {}, (224,224))]
rows = []
for name, mn, kw, hw in M:
    m = timm.create_model(mn, pretrained=False, num_classes=C, **kw).eval()
    x = torch.randn(1, 3, *hw)
    for _ in range(10): m(x)
    ts = []
    for _ in range(50):
        t0 = time.perf_counter(); m(x); ts.append((time.perf_counter()-t0)*1000)
    rows.append((name, round(sum(p.numel() for p in m.parameters())/1e6, 2), round(float(np.median(ts)), 1)))
R = pd.DataFrame(rows, columns=['model', 'params (M)', 'median latency (ms)']); display(R)
print('Note: the MoE head of AmsiNet-T adds 1.3 M parameters; its latency was 0.3 ms above linear on the earlier host.')